## chargement de la table intégrée et outils de calcul

On charge `agg_data/zones_integre.csv` (263 zones, 92 colonnes de totaux) et on crée la table `profil` avec les 4 colonnes d'identification : `zone_id`, `nom_zone`, `arrondissement`, `type_zone`. Les features y seront ajoutées une famille après l'autre.

On définit deux petits outils :
- `ratio(numérateur, dénominateur)` : divise, et renvoie une valeur vide (NaN) quand le dénominateur vaut 0, au lieu de provoquer une division par zéro ;
- `somme(colonne, services)` : additionne la même colonne de plusieurs services, par exemple `s_prix` de Yellow, Green et HVFHV.

On définit aussi les groupes de services : `TOUS` (les 4), `TAXIS` (Yellow et Green, les seuls avec un moyen de paiement), `AVEC_PRIX` (Yellow, Green et HVFHV, les seuls avec prix, distance et pourboire). La période compte **119 jours**.

**Effet sur les données :** une table `profil` de 263 lignes et 4 colonnes, prête à recevoir les features.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")

AGG_DIR = os.path.join(os.getcwd(), "agg_data")
NB_JOURS = 119

integre = pd.read_csv(os.path.join(AGG_DIR, "zones_integre.csv"))
print("Lignes :", integre.shape[0], "| colonnes :", integre.shape[1])

profil = integre[["zone_id", "nom_zone", "arrondissement", "type_zone"]].copy()

def ratio(numerateur, denominateur):
    return (numerateur / denominateur.replace(0, np.nan)).astype("float64")

def somme(nom_colonne, services):
    return sum(integre[s + "_" + nom_colonne] for s in services)

TOUS = ["yellow", "green", "fhv", "hvfhv"]
TAXIS = ["yellow", "green"]
AVEC_PRIX = ["yellow", "green", "hvfhv"]
display(profil.head())

Lignes : 263 | colonnes : 92


,zone_id,nom_zone,arrondissement,type_zone
0,1,Newark Airport,EWR,EWR
1,2,Jamaica Bay,Queens,Boro Zone
2,3,Allerton/Pelham Gardens,Bronx,Boro Zone
3,4,Alphabet City,Manhattan,Yellow Zone
4,5,Arden Heights,Staten Island,Boro Zone


## features d'activité (5)

- `courses_par_jour` = `total_n / 119` : le volume moyen de courses par jour, tous services confondus. C'est la mesure de l'intensité de la demande dans la zone.
- `part_yellow` = `yellow_n / total_n` : la part des taxis jaunes.
- `part_green` = `green_n / total_n` : la part des taxis verts.
- `part_fhv` = `fhv_n / total_n` : la part des VTC classiques.
- `part_vtc` = `hvfhv_n / total_n` : la part des plateformes (Uber, Lyft…).

Les 4 parts décrivent le **mix de services** de la zone : qui transporte les gens ici. On vérifie que leur somme vaut 1 pour chaque zone.

**Effet sur les données :** 5 colonnes de plus. On affiche les 10 zones les plus actives.

In [2]:
profil["courses_par_jour"] = integre["total_n"] / NB_JOURS
profil["part_yellow"] = ratio(integre["yellow_n"], integre["total_n"])
profil["part_green"] = ratio(integre["green_n"], integre["total_n"])
profil["part_fhv"] = ratio(integre["fhv_n"], integre["total_n"])
profil["part_vtc"] = ratio(integre["hvfhv_n"], integre["total_n"])

verif = profil[["part_yellow", "part_green", "part_fhv", "part_vtc"]].sum(axis=1)
print("Somme des 4 parts = 1 :", np.allclose(verif[integre["total_n"] > 0], 1))
display(profil.sort_values("courses_par_jour", ascending=False).head(10))

Somme des 4 parts = 1 : True


,zone_id,nom_zone,arrondissement,type_zone,courses_par_jour,part_yellow,part_green,part_fhv,part_vtc
160,161,Midtown Center,Manhattan,Yellow Zone,5957.268908,0.922436,0.000013,0.010671,0.066880
236,237,Upper East Side South,Manhattan,Yellow Zone,5555.201681,0.944933,0.000014,0.006845,0.048208
131,132,JFK Airport,Queens,Airports,5552.588235,0.865421,0.000121,0.030000,0.104457
235,236,Upper East Side North,Manhattan,Yellow Zone,5169.638655,0.944644,0.002167,0.006536,0.046652
229,230,Times Sq/Theatre District,Manhattan,Yellow Zone,4497.521008,0.890550,0.000015,0.018367,0.091068
161,162,Midtown East,Manhattan,Yellow Zone,4280.907563,0.927236,0.000020,0.010702,0.062042
185,186,Penn Station/Madison Sq West,Manhattan,Yellow Zone,4109.142857,0.944279,0.000016,0.004642,0.051063
137,138,LaGuardia Airport,Queens,Airports,3980.596639,0.819123,0.000171,0.024995,0.155711
141,142,Lincoln Square East,Manhattan,Yellow Zone,3899.285714,0.935349,0.000026,0.006927,0.057699
233,234,Union Sq,Manhattan,Yellow Zone,3741.260504,0.895521,0.000013,0.007329,0.097136


## features de rythme (6)

- `part_weekend` : courses du samedi et du dimanche des 4 services, divisées par le total. Une zone « neutre » serait autour de 0,30, car la période compte 36 jours de week-end sur 119 ; en dessous, la zone vit au rythme du travail, au-dessus, au rythme des loisirs.
- `part_nuit`, `part_matin`, `part_journee`, `part_fin_aprem`, `part_soiree` : part des courses de chaque tranche horaire, tous services confondus.

Ces 6 variables décrivent le **profil temporel** de la zone (quartier d'affaires, résidentiel, de sortie…). On vérifie que les 5 tranches font 1 pour chaque zone.

**Effet sur les données :** 6 colonnes de plus.

In [3]:
profil["part_weekend"] = ratio(somme("n_weekend", TOUS), integre["total_n"])
for tranche in ["nuit", "matin", "journee", "fin_aprem", "soiree"]:
    profil["part_" + tranche] = ratio(somme("n_" + tranche, TOUS), integre["total_n"])

tranches = ["part_nuit", "part_matin", "part_journee", "part_fin_aprem", "part_soiree"]
print("Somme des 5 tranches = 1 :", np.allclose(profil.loc[integre["total_n"] > 0, tranches].sum(axis=1), 1))
display(profil[["zone_id", "nom_zone", "part_weekend"] + tranches].head(10))

Somme des 5 tranches = 1 : False


,zone_id,nom_zone,part_weekend,part_nuit,part_matin,part_journee,part_fin_aprem,part_soiree
0,1,Newark Airport,0.273595,0.080929,0.100022,0.307659,0.102842,0.207203
1,2,Jamaica Bay,0.433333,0.100000,0.133333,0.333333,0.266667,0.033333
2,3,Allerton/Pelham Gardens,0.274229,0.087412,0.236653,0.329662,0.097628,0.146842
3,4,Alphabet City,0.523277,0.333227,0.139660,0.161903,0.121392,0.216606
4,5,Arden Heights,0.195688,0.059761,0.396063,0.373096,0.026014,0.048277
5,6,Arrochar/Fort Wadsworth,0.166026,0.052748,0.289777,0.446964,0.024212,0.062164
6,7,Astoria,0.367953,0.147278,0.186737,0.269736,0.121556,0.186414
7,8,Astoria Park,0.317972,0.087558,0.267281,0.308756,0.055300,0.152074
8,9,Auburndale,0.241047,0.077457,0.234422,0.394857,0.069254,0.096387
9,10,Baisley Park,0.277520,0.095649,0.286859,0.313062,0.107511,0.127030


## caractéristiques des courses (5)

- `duree_moy_min` : somme des durées valides des 4 services / nombre de durées valides. FHV y participe, car c'est sa seule mesure.
- `distance_moy` : somme des distances valides / nombre de distances valides, sur Yellow, Green et HVFHV (FHV n'a pas de distance).
- `prix_moy` : somme des prix valides / nombre de prix valides, sur les mêmes 3 services.
- `prix_par_mile` : `s_prix_pm / s_dist_pm`, calculé uniquement sur les courses où le prix **et** la distance sont valides.
- `part_aeroport` : courses qui finissent à Newark, JFK ou LaGuardia, sur le total des 4 services.

Chaque moyenne est **pondérée** par le nombre de courses de chaque service : un service très présent dans la zone pèse davantage, ce qui donne la vraie moyenne de la zone.

**Effet sur les données :** 5 colonnes de plus. On affiche leurs statistiques descriptives.

In [4]:
profil["duree_moy_min"] = ratio(somme("s_duree", TOUS), somme("n_duree", TOUS))
profil["distance_moy"] = ratio(somme("s_dist", AVEC_PRIX), somme("n_dist", AVEC_PRIX))
profil["prix_moy"] = ratio(somme("s_prix", AVEC_PRIX), somme("n_prix", AVEC_PRIX))
profil["prix_par_mile"] = ratio(somme("s_prix_pm", AVEC_PRIX), somme("s_dist_pm", AVEC_PRIX))
profil["part_aeroport"] = ratio(somme("n_aeroport", TOUS), integre["total_n"])

display(profil[["zone_id", "nom_zone", "duree_moy_min", "distance_moy", "prix_moy", "prix_par_mile", "part_aeroport"]].describe().T.round(2))

,count,mean,std,min,25%,50%,75%,max
zone_id,263.0,132.00,76.07,1.00,66.50,132.00,197.50,263.00
duree_moy_min,261.0,19.97,4.37,9.88,17.79,19.68,21.80,42.10
distance_moy,261.0,4.67,1.61,0.80,4.02,4.56,5.22,15.96
prix_moy,261.0,22.75,6.77,11.18,19.92,21.66,24.16,89.07
prix_par_mile,261.0,5.09,1.18,3.65,4.33,4.86,5.56,15.03
part_aeroport,261.0,0.03,0.03,0.00,0.01,0.02,0.04,0.28


## paiement et péage CBD (3)

- `taux_pourboire` : somme des pourboires / somme des prix correspondants, sur les paiements par carte des taxis et sur toutes les courses VTC. Ce sont les seules courses dont le pourboire est réellement enregistré.
- `part_especes` : paiements en espèces / paiements connus (carte ou espèces), uniquement pour Yellow et Green, car les VTC sont toujours payés dans l'application.
- `part_cbd` : courses de 2025 qui ont payé le péage / courses de 2025, sur Yellow, Green et HVFHV. On divise par les courses de 2025 car le péage n'existait pas en 2024. FHV n'a pas de colonne de péage, il n'entre donc ni au numérateur ni au dénominateur.

**Effet sur les données :** 3 colonnes de plus. On affiche les 10 zones les plus exposées au péage.

In [5]:
profil["taux_pourboire"] = ratio(somme("s_pourboire", AVEC_PRIX), somme("s_prix_pb", AVEC_PRIX))
profil["part_especes"] = ratio(somme("n_especes", TAXIS), somme("n_paiement", TAXIS))
profil["part_cbd"] = ratio(somme("n_cbd", AVEC_PRIX), somme("n_2025", AVEC_PRIX))

display(profil[["taux_pourboire", "part_especes", "part_cbd"]].describe().T.round(3))
display(profil.sort_values("part_cbd", ascending=False)[["zone_id", "nom_zone", "arrondissement", "part_cbd"]].head(10))

,count,mean,std,min,25%,50%,75%,max
taux_pourboire,261.0,0.070,0.066,0.0,0.024,0.044,0.084,0.231
part_especes,258.0,0.180,0.159,0.0,0.088,0.135,0.214,1.000
part_cbd,261.0,0.235,0.335,0.0,0.032,0.059,0.240,1.000


,zone_id,nom_zone,arrondissement,part_cbd
104,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,1.000000
198,199,Rikers Island,Bronx,1.000000
210,211,SoHo,Manhattan,0.986662
248,249,West Village,Manhattan,0.986476
223,224,Stuy Town/Peter Cooper Village,Manhattan,0.986381
113,114,Greenwich Village South,Manhattan,0.985539
112,113,Greenwich Village North,Manhattan,0.985438
233,234,Union Sq,Manhattan,0.985398
78,79,East Village,Manhattan,0.984948
143,144,Little Italy/NoLiTa,Manhattan,0.984394


## features propres aux VTC (4)

Ces variables ne viennent que de HVFHV, le seul service qui fournit ces informations :
- `attente_moy_min` = `s_attente / n_attente` : temps moyen entre la commande et la prise en charge, indicateur de la **disponibilité** des VTC dans la zone ;
- `part_partage` = `n_partage / n_partage_ok` : part des courses partagées demandées ;
- `part_pmr` = `n_pmr / n_pmr_ok` : part des demandes de véhicule accessible aux fauteuils roulants ;
- `part_chauffeur` = `s_paie / s_prix_paie` : part du prix payé par le passager qui revient au chauffeur.

**Effet sur les données :** 4 colonnes de plus.

In [6]:
profil["attente_moy_min"] = ratio(integre["hvfhv_s_attente"], integre["hvfhv_n_attente"])
profil["part_partage"] = ratio(integre["hvfhv_n_partage"], integre["hvfhv_n_partage_ok"])
profil["part_pmr"] = ratio(integre["hvfhv_n_pmr"], integre["hvfhv_n_pmr_ok"])
profil["part_chauffeur"] = ratio(integre["hvfhv_s_paie"], integre["hvfhv_s_prix_paie"])

display(profil[["attente_moy_min", "part_partage", "part_pmr", "part_chauffeur"]].describe().T.round(3))

,count,mean,std,min,25%,50%,75%,max
attente_moy_min,259.0,4.883,0.949,3.547,4.340,4.643,5.096,11.088
part_partage,259.0,0.040,0.019,0.000,0.030,0.038,0.048,0.200
part_pmr,259.0,0.003,0.002,0.000,0.002,0.002,0.004,0.012
part_chauffeur,259.0,0.794,0.058,0.632,0.770,0.812,0.827,1.134


## évolution 2024 -> 2025 (1)

- `croissance_2025` = (courses 2025 / courses 2024) − 1, tous services confondus. Une valeur de 0,10 veut dire +10 % de courses en un an, −0,20 veut dire −20 %.

Les deux périodes couvrent les mêmes mois (février-mars), donc la comparaison n'est pas faussée par la saison. Elle permet de voir quelles zones ont gagné ou perdu des courses après l'arrivée du péage CBD. On affiche la croissance globale de la ville, puis les 5 zones qui ont le plus baissé et les 5 qui ont le plus augmenté.

**Effet sur les données :** 1 colonne de plus, soit 24 features au total.

In [7]:
n_2024 = somme("n_2024", TOUS)
n_2025 = somme("n_2025", TOUS)
profil["croissance_2025"] = ratio(n_2025, n_2024) - 1

print("Croissance globale de la ville :", round(n_2025.sum() / n_2024.sum() - 1, 4))
display(profil["croissance_2025"].describe().to_frame().T.round(3))
display(profil.sort_values("croissance_2025")[["zone_id", "nom_zone", "croissance_2025"]].head(5))
display(profil.sort_values("croissance_2025")[["zone_id", "nom_zone", "croissance_2025"]].tail(5))

Croissance globale de la ville : 0.1265


,count,mean,std,min,25%,50%,75%,max
croissance_2025,260.0,0.165,0.168,-0.6,0.074,0.167,0.253,1.0


,zone_id,nom_zone,croissance_2025
109,110,Great Kills Park,-0.600000
55,56,Corona,-0.413580
1,2,Jamaica Bay,-0.333333
72,73,East Flushing,-0.293866
172,173,North Corona,-0.274658


,zone_id,nom_zone,croissance_2025
193,194,Randalls Island,0.840149
198,199,Rikers Island,1.000000
102,103,Governor's Island/Ellis Island/Liberty Island,NaN
103,104,Governor's Island/Ellis Island/Liberty Island,NaN
104,105,Governor's Island/Ellis Island/Liberty Island,NaN


## zones à faible activité et valeurs vides

Une zone avec moins d'**une course par jour** (moins de 119 courses sur la période) n'a pas assez de courses pour que ses moyennes et ses parts soient fiables : un seul trajet inhabituel suffirait à les déformer. Ce sont surtout des parcs, des cimetières, des îles ou des zones industrielles.

On crée la colonne `faible_activite` (1 si la zone est sous le seuil, 0 sinon). Ces zones restent dans le fichier, mais elles seront **exclues du clustering** pour ne pas créer de faux segments.

On vérifie ensuite les valeurs vides des 24 features dans les zones actives. Une valeur vide apparaît quand le dénominateur vaut 0, par exemple une zone sans aucune course HVFHV n'a pas d'attente moyenne. Elles seront traitées à l'étape de préparation.

**Effet sur les données :** 1 colonne de plus, `faible_activite`.

In [8]:
SEUIL_COURSES_JOUR = 1
profil["faible_activite"] = (profil["courses_par_jour"] < SEUIL_COURSES_JOUR).astype("int64")

print("Zones à faible activité (< 1 course par jour) :", profil["faible_activite"].sum())
display(profil.loc[profil["faible_activite"] == 1, ["zone_id", "nom_zone", "arrondissement", "courses_par_jour"]])

FEATURES = [c for c in profil.columns if c not in ["zone_id", "nom_zone", "arrondissement", "type_zone", "faible_activite"]]
print("Nombre de features :", len(FEATURES))

actives = profil[profil["faible_activite"] == 0]
vides = actives[FEATURES].isna().sum()
print("Valeurs vides dans les zones actives :")
display(vides[vides > 0].to_frame("nb_vides"))

Zones à faible activité (< 1 course par jour) : 6


,zone_id,nom_zone,arrondissement,courses_par_jour
1,2,Jamaica Bay,Queens,0.252101
102,103,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.000000
103,104,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.000000
104,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.008403
109,110,Great Kills Park,Staten Island,0.058824
198,199,Rikers Island,Bronx,0.075630


Nombre de features : 24
Valeurs vides dans les zones actives :


,nb_vides
part_especes,1
attente_moy_min,1
part_partage,1
part_pmr,1
part_chauffeur,1


## description et sauvegarde du profil des zones

On affiche les statistiques descriptives des 24 features (moyenne, écart-type, minimum, quartiles, maximum) pour vérifier que les valeurs sont plausibles : des parts entre 0 et 1, des durées et des prix réalistes.

Puis on sauvegarde le **dataset final** dans `agg_data/zones_profil.csv`.

**Effet sur les données :** aucun changement. Le fichier contient 263 lignes (une par zone) et 29 colonnes : 4 d'identification, 24 features et l'indicateur `faible_activite`. C'est ce fichier qui servira à la préparation (imputation, transformation, standardisation) puis au clustering.

In [9]:
display(profil[FEATURES].describe().T.round(3))

cible = os.path.join(AGG_DIR, "zones_profil.csv")
profil.to_csv(cible, index=False)
print("Sauvegardé :", cible)
print("Lignes :", profil.shape[0], "| colonnes :", profil.shape[1])

,count,mean,std,min,25%,50%,75%,max
courses_par_jour,263.0,625.367,1122.931,0.000,90.958,179.042,425.950,5957.269
part_yellow,261.0,0.308,0.306,0.000,0.098,0.168,0.444,0.945
part_green,261.0,0.014,0.038,0.000,0.000,0.004,0.008,0.379
part_fhv,261.0,0.182,0.201,0.000,0.051,0.110,0.222,0.837
part_vtc,261.0,0.496,0.255,0.000,0.246,0.565,0.703,1.000
part_weekend,261.0,0.304,0.083,0.139,0.259,0.298,0.332,1.000
part_nuit,261.0,0.099,0.054,0.000,0.068,0.094,0.115,0.383
part_matin,261.0,0.194,0.070,0.000,0.144,0.205,0.243,0.396
part_journee,261.0,0.322,0.079,0.000,0.281,0.312,0.354,0.658
part_fin_aprem,261.0,0.133,0.074,0.000,0.089,0.108,0.174,0.328


Sauvegardé : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\agg_data\zones_profil.csv
Lignes : 263 | colonnes : 29
